# DentalCare Account Manager - Colab Version

## Optimized for 90% Resource Efficiency

**Performance Stats:**
- Per Chrome Instance: ~160-180MB RAM
- Colab Free Tier: ~12.9GB RAM
- Recommended Workers: 50 (uses ~9GB, leaves headroom)

## Usage

1. Just set your **Phone API URL** below
2. Run all cells
3. App auto-installs and starts

In [ ]:
# ============================================
# PHONE API URL - THIS IS THE ONLY THING YOU NEED TO SET
# ============================================

# Your MFA phone service URL (where phone numbers are returned)
MFA_PHONE_URL = "http://localhost:6729/get-number"  # @param {type:"string"}

# Optional settings (defaults work well for 90% efficiency)
PARALLEL_COUNT = 50  # @param {type:"integer"}  (50 for 90% efficiency on 12.9GB RAM)
BATCH_COUNT = 0  # @param {type:"integer"}  (0 = unlimited)

print(f"Configuration:")
print(f"  Phone API: {MFA_PHONE_URL}")
print(f"  Parallel Workers: {PARALLEL_COUNT}")
print(f"  Batch Count: {'Unlimited' if BATCH_COUNT == 0 else BATCH_COUNT}")
print(f"  RAM Usage: ~{PARALLEL_COUNT * 180}MB (~{PARALLEL_COUNT * 180 / 1024:.1f}GB)")

## Step 1: Install Dependencies

In [ ]:
!pip install playwright requests > /dev/null 2>&1
!playwright install chromium > /dev/null 2>&1
print("Dependencies installed!")

## Step 2: Clone Repository

In [ ]:
import os

# Hardcoded GitHub URL
GITHUB_REPO_URL = "https://github.com/sulaiman282/Colab_playwrite_test"

repo_name = GITHUB_REPO_URL.rstrip('/').split('/')[-1]
app_dir = f'/content/{repo_name}'

print(f"Cloning {GITHUB_REPO_URL}...")

# Remove existing directory if present
if os.path.exists(app_dir):
    !rm -rf {app_dir}

# Clone the repository
!git clone {GITHUB_REPO_URL} {app_dir} 2>&1

print(f"Repository cloned to {app_dir}")

## Step 3: Run the Account Manager

In [ ]:
import sys
sys.path.insert(0, app_dir)

os.chdir(app_dir)

from run import ColabRunner

runner = ColabRunner(
    parallel_count=PARALLEL_COUNT,
    batch_count=BATCH_COUNT,
    phone_api_url=MFA_PHONE_URL
)

print("\n" + "="*60)
print("STARTING DENTALCARE ACCOUNT MANAGER")
print("="*60)
print(f"Workers: {PARALLEL_COUNT} | RAM: ~{PARALLEL_COUNT * 180 / 1024:.1f}GB")
print("Press STOP to interrupt\n")

loop = asyncio.new_event_loop()
asyncio.set_event_loop(loop)
try:
    loop.run_until_complete(runner.run())
finally:
    loop.close()

## Step 4: Download Results

In [ ]:
import glob

results_dir = f'{app_dir}/data/results'
result_files = glob.glob(f'{results_dir}/accounts_*.json')

if result_files:
    latest_file = max(result_files, key=os.path.getmtime)
    print(f"Results: {latest_file}")
    print(f"\nDownload:")
    print(f"from google.colab import files")
    print(f"files.download('{latest_file}')")
else:
    print("No results found.")